# SatyaVoice — Kaggle Development / Benchmarking Notebook

**DEVELOPMENT/TESTING ONLY.** Runs the *same shared pipeline* as production
(`hf_zero_gpu/inference.py` via `app/services.local_provider`) on Kaggle GPU.
HF ZeroGPU remains the production provider. Kaggle must not stay online and
must never receive user audio.

Sections: 1 env check · 2 GPU check · 3 repo setup · 4 deps · 5 config ·
6 checkpoints · 7 preprocessing sanity · 8 single inference · 9 batch eval ·
10 latency · 11 memory · 12 metrics · 13 provider consistency · 14 export.

In [ ]:
# ===== 1+2. Environment & GPU check =====
# Prints whatever GPU Kaggle assigned (never hard-coded), CUDA/PyTorch
# versions and available VRAM.
import subprocess, sys, os
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU:', p.name)
    print('VRAM total: %.1f MB' % (p.total_memory / 1024**2))
    free, _total = torch.cuda.mem_get_info()
    print('VRAM free:   %.1f MB' % (free / 1024**2))
assert torch.cuda.is_available(), 'Kaggle GPU accelerator is NOT enabled.'

In [ ]:
# ===== 3+4. Repository setup & dependencies =====
import os
if not os.path.exists('sih-voice'):
    !git clone https://github.com/Devanshshar01/sih-voice.git
%cd sih-voice
!pip install -q -r kaggle/requirements-kaggle.txt

In [ ]:
# ===== 5. Configuration (PRODUCTION checkpoint contract) =====
# Set the SAME fine-tuned checkpoint the HF Space uses. The default base
# facebook/wav2vec2-xls-r-300m is NOT the SatyaVoice anti-spoof model.
os.environ.setdefault('INFERENCE_PROVIDER', 'local')  # 'kaggle' is an alias
os.environ.setdefault('DEVICE', 'cuda')
os.environ.setdefault('SPEAKER_MODEL_ID', 'speechbrain/spkrec-ecapa-voxceleb')
# os.environ['ANTISPOOF_MODEL_ID'] = '<fine-tuned SatyaVoice checkpoint>'
from hf_zero_gpu import config
print('antispoof checkpoint:', config.ANTISPOOF_MODEL_ID)
print('speaker checkpoint  :', config.SPEAKER_MODEL_ID)
if config.ANTISPOOF_MODEL_ID == 'facebook/wav2vec2-xls-r-300m':
    print('*** BASE XLS-R — NOT FINAL SATYAVOICE ANTI-SPOOF MODEL ***')

In [ ]:
# ===== 6+7. Checkpoint loading & preprocessing sanity =====
import numpy as np
from hf_zero_gpu import inference as pipeline
pipeline._load_models()
pipeline._ensure_models_on_device()
print('device:', pipeline.DEVICE)

# Preprocessing contract: mono -> REAL resample to 16 kHz -> exactly 4 s
# (64000 samples). Test with an off-contract input (44.1 kHz stereo).
sr = 44100
t = np.arange(sr * 5) / sr
stereo = np.stack([0.4*np.sin(2*np.pi*220*t)]*2, axis=1).astype(np.float32)
prepared = pipeline._prepare_audio(stereo)
print('prepared length:', prepared.shape[0], '(expected 64000)')
assert prepared.shape[0] == 64000
print('preprocessing sanity: PASS')

In [ ]:
# ===== 8. Single-sample sanity inference (structured, no fakes) =====
from app.services.local_provider import LocalInferenceProvider
provider = LocalInferenceProvider(env=dict(os.environ))
result = provider.infer_audio_window_sync(prepared)
print('=' * 50)
print('SatyaVoice inference')
print('=' * 50)
print('provider:               ', result.provider)
print('sample_rate:            ', result.sample_rate)
print('duration_ms:            ', result.duration_ms)
print('success:                ', result.success)
if result.success:
    print('spoof_probability:       %.4f' % result.spoof_probability)
    print('speaker_embedding_dim:  ', len(result.speaker_embedding))
print('model_version_antispoof:', result.model_version_antispoof)
print('model_version_speaker:  ', result.model_version_speaker)
print('inference_time_ms:      ', result.inference_time_ms)
print('=' * 50)

In [ ]:
# ===== 9+10+11. Batch / latency / memory benchmark =====
# 100+ runs after warm-up, CUDA-synchronised, p50/p95/p99 for XLS-R / ECAPA
# / combined critical path + VRAM; writes JSON+CSV to kaggle/reports/.
!python kaggle/benchmark.py --mode benchmark --runs 100

In [ ]:
# ===== 12. Classification metrics on labelled data (OPTIONAL) =====
# Ground truth from filename prefixes: fake_/spoof_ vs real_/genuine_/bonafide_.
# Language prefixes hi_/ta_/te_/bn_/mr_/en_ enable per-language metrics.
# Without labels the run is reported as QUALITATIVE -- no metrics invented.
# !python kaggle/benchmark.py --mode evaluate --data-dir /kaggle/working/eval_wavs

In [ ]:
# ===== 13. Cross-provider consistency: Kaggle vs HF ZeroGPU =====
# Same deterministic samples through both providers; tolerance-based drift
# check on scores, verdict agreement and model-version agreement.
!python kaggle/consistency.py --space https://devanshshar01-satyavoice-gpu.hf.space --samples 5 --tolerance 0.02

In [ ]:
# ===== 14. Export / reporting =====
# Benchmark & consistency runs write JSON+CSV to kaggle/reports/.
# Committing large audio or private recordings is NOT allowed.
import glob, json
for p in sorted(glob.glob('kaggle/reports/*.json')):
    print(p)
    print(json.dumps(json.load(open(p)), indent=2)[:1200])